In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import os

def process_airline_data(file_path):
    df = pd.read_csv(file_path)
    
    # Ensure Year column is integer for slider to work correctly
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(0).astype(int)
    
    # Data Cleansing & Trasnformation
    # Convert thousands using commas (string) to numbers (float)
    cols_to_fix = ['Dom_Pax', 'Int_Pax', 'Dom_RPM', 'Int_RPM', 'Dom_ASM', 'Int_ASM']
    for col in cols_to_fix:
        if df[col].dtype == 'object':
            df[col] = df[col].str.replace(',', '').astype(float)
            
    # Remove COVID-19 anomaly (Year 2020)
    df = df[df['Year'] != 2020].copy()

    # Check Year 2021: Is performance always below average? This is because 2021 is the begininng of new normal
    # (i) Calculate baseline average (excluding year 2021)
    df_baseline = df[df['Year'] != 2021]
    baseline_avg = df_baseline.groupby('Month')[['Dom_LF', 'Int_LF']].mean().reset_index()
    
    # (ii)   Compare with 2021 data
    df_2021 = df[df['Year'] == 2021]
    
    if not df_2021.empty:
        merged = pd.merge(df_2021, baseline_avg, on='Month', suffixes=('', '_avg'))
        
        # Check condition: Is performance (Domestic & International) above average for at least 6 months in 2021?
        dom_months_above = (merged['Dom_LF'] > merged['Dom_LF_avg']).sum()
        int_months_above = (merged['Int_LF'] > merged['Int_LF_avg']).sum()
        
        if dom_months_above >= 6 and int_months_above >= 6:
            print(f"Info: Year 2021 included (Domestic: {dom_months_above} months > avg, International: {int_months_above} months > avg).")
        else:
            print(f"Info: Year 2021 excluded (Domestic: {dom_months_above} months > avg, International: {int_months_above} months > avg). Both need >= 6.")
            df = df[df['Year'] != 2021]
    
    return df

def apply_dynamic_pricing(df):
    """
    Pricing logic based on existing Load Factor (LF) columns.
    Peak Season: If LF is above 80%
    Normal Season: If LF is in between 75-80%
    Low Season: If LF is below 75%
    """
    recommendations = []
    
    for index, row in df.iterrows():
        # Using Dom_LF column from your data
        if row['Dom_LF'] > 80.0:
            strategy = "PEAK: Increase Base Fare"
        elif row['Dom_LF'] < 75.0:
            strategy = "LOW: Seasonal Discount"
        else:
            strategy = "NORMAL: Standard Pricing"
        
        recommendations.append(strategy)
    
    df['Strategy_Recommendation'] = recommendations
    return df